# Conversation Simulator & DB Utilities

This notebook helps you:
- Simulate a conversation between a companion and a synthetic user for N turns
- Persist the conversation (conversations/messages) to Postgres
- Optionally create memory items from the messages via the API (with embeddings + importance eval)
- Delete a conversation and its full footprint (messages and linked memories)

Requirements: OPENAI_API_KEY set in environment; DATABASE_DSN set for DB access.
If the API runs with DISABLE_AUTH_FOR_TESTING=true, we can fetch a dev token automatically.

In [ ]:
# Setup & imports
import asyncio
import json
import os
import subprocess
import sys
from typing import Any, Dict, List


# Attempt to ensure deps using uv (preferred) if missing; otherwise, print guidance.
def _ensure(package: str, import_name: str | None = None):
    name = import_name or package
    try:
        __import__(name)
        return
    except Exception:
        pass
    try:
        # Install into the current kernel's Python using uv
        subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, package])
        __import__(name)
    except Exception:
        print(f"Could not import or install {package}. If using uv, run:")
        print(f"  uv pip install --python {sys.executable} {package}")
        raise


for pkg, imp in [
    ("requests", "requests"),
    ("openai", "openai"),
    ("asyncpg", "asyncpg"),
    ("python-dotenv", "dotenv"),
    ("nest_asyncio", "nest_asyncio"),
]:
    _ensure(pkg, imp)

import requests
from dotenv import load_dotenv

load_dotenv()

import nest_asyncio as _nest_asyncio


def arun(coro):
    """Run an async coroutine safely in notebooks and scripts.

    - In Jupyter (running loop), applies nest_asyncio and uses run_until_complete.
    - Otherwise, falls back to asyncio.run.
    """
    try:
        loop = asyncio.get_running_loop()
        try:
            _nest_asyncio.apply()
        except Exception:
            pass
        return loop.run_until_complete(coro)
    except RuntimeError:
        return asyncio.run(coro)


API = os.getenv("API_BASE", "http://localhost:8100")
COMPANION_ID = "456b025e-b160-482a-be91-250fe821ff67"  # fill if desired
EXTERNAL_USER_ID = "ccbeb23d-f72f-4ffb-b3cd-84945c23cc30"  # os.getenv('TEST_EXTERNAL_USER_ID', f'sim-user-{uuid.uuid4()}')
TOKEN = os.getenv("TEST_API_TOKEN", "mock-dev-token")

# Try dev token if server allows
try:
    r = requests.get(f"{API}/api/auth/dev-token", timeout=3)
    if r.ok and isinstance(r.json(), dict) and r.json().get("token"):
        TOKEN = r.json()["token"]
        print("Using dev token from API")
except Exception:
    pass

print("API =", API)
print("COMPANION_ID =", COMPANION_ID)
print("EXTERNAL_USER_ID =", EXTERNAL_USER_ID)
print("TOKEN set =", bool(TOKEN))

In [ ]:
# HTTP helpers
def api_request(
    path: str,
    method: str = "GET",
    json_body: Dict[str, Any] | None = None,
    params: Dict[str, Any] | None = None,
):
    url = f"{API}{path}"
    headers = {"Content-Type": "application/json"}
    if TOKEN:
        headers["Authorization"] = f"Bearer {TOKEN}"
    resp = requests.request(
        method.upper(), url, headers=headers, json=json_body, params=params, timeout=60
    )
    if not resp.ok:
        raise RuntimeError(f"{method} {path} -> {resp.status_code}: {resp.text[:2000]}")
    try:
        return resp.json()
    except Exception:
        return resp.text


def get_companion_config(companion_id: str) -> Dict[str, Any]:
    cfg = api_request(f"/api/companions/{companion_id}", "GET")
    return cfg

In [ ]:
# Conversation simulation (self-play)
from openai import OpenAI

_client = None


def openai_client() -> OpenAI:
    global _client
    if _client is None:
        _client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    return _client


def simulate_conversation(
    companion_id: str,
    user_profile: str,
    seed_user_message: str = "Hi there!",
    turns: int = 6,
    model: str = "gpt-4o-mini",
    temperature: float = 0.7,
) -> List[Dict[str, str]]:
    cfg = get_companion_config(companion_id)
    sys_prompt = cfg.get("system_prompt", {}).get("full_system_prompt", "")
    if not sys_prompt:
        raise ValueError("Companion system prompt not found")
    cli = openai_client()
    convo: List[Dict[str, str]] = []
    # Start with a user seed
    convo.append({"role": "system", "content": sys_prompt})
    print("System prompt:", sys_prompt)
    convo.append({"role": "user", "content": seed_user_message})

    # Helper to call model
    def _chat(msgs):
        r = cli.chat.completions.create(
            model=model, messages=msgs, temperature=temperature, max_tokens=400
        )
        return r.choices[0].message.content.strip()

    # Generate assistant reply
    assistant = _chat(convo)
    convo.append({"role": "assistant", "content": assistant})
    # For each remaining turn, synthesize user and assistant
    for _ in range(max(0, turns - 1)):
        # Synthesize next user turn based on a user persona
        user_sim_msgs = [
            {
                "role": "system",
                "content": f"You simulate the USER in a dialogue. Persona: {user_profile}. Reply briefly (1-2 sentences).",
            },
            {
                "role": "user",
                "content": "Given the conversation so far (below), produce the next user message only.",
            },
            {"role": "user", "content": json.dumps(convo, ensure_ascii=False)},
        ]
        user_next = _chat(user_sim_msgs)
        convo.append({"role": "user", "content": user_next})
        # Assistant reply using the companion system prompt
        assistant = _chat(convo)
        convo.append({"role": "assistant", "content": assistant})
    # Strip initial system before persisting
    convo_no_system = [m for m in convo if m["role"] != "system"]
    return convo_no_system

In [ ]:
# DB persistence utilities (synchronous; psycopg2)
DB_DSN = os.getenv("DATABASE_DSN") or os.getenv("DATABASE_TRANSACTION_DSN")
if not DB_DSN:
    print("Warning: DATABASE_DSN not set; DB operations will fail")

try:
    import psycopg2
except Exception:
    try:
        subprocess.check_call(
            ["uv", "pip", "install", "--python", sys.executable, "psycopg2-binary"]
        )
        import psycopg2
    except Exception:
        print("Could not import psycopg2. Install manually:")
        print(f"  uv pip install --python {sys.executable} psycopg2-binary")
        raise


def persist_conversation(
    companion_id: str, external_user_id: str, messages: List[Dict[str, str]]
) -> str:
    conn = psycopg2.connect(DB_DSN)
    try:
        conn.autocommit = False
        with conn.cursor() as cur:
            cur.execute(
                "INSERT INTO conversations (companion_id, external_user_id) VALUES (%s::uuid,%s) RETURNING id",
                (companion_id, external_user_id),
            )
            conversation_id = str(cur.fetchone()[0])
            for m in messages:
                role = m.get("role")
                content = m.get("content")
                if not role or not content:
                    continue
                cur.execute(
                    "INSERT INTO messages (conversation_id, role, content) VALUES (%s::uuid,%s,%s)",
                    (conversation_id, role, content),
                )
        conn.commit()
        return conversation_id
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def create_memories_from_messages(
    companion_id: str, conversation_id: str, messages: List[Dict[str, str]]
):
    # Create memories via API so embeddings + importance are computed server-side
    for m in messages:
        role = (m.get("role") or "").lower()
        if role not in ("user", "assistant"):
            continue
        body = {
            "content": m.get("content", ""),
            "sender_type": role,
            "conversation_id": conversation_id,
        }
        try:
            api_request(f"/api/companions/{companion_id}/memories", "POST", json_body=body)
        except Exception as e:
            print("Failed to create memory for message:", e)


def delete_conversation_footprint(conversation_id: str):
    conn = psycopg2.connect(DB_DSN)
    try:
        conn.autocommit = False
        with conn.cursor() as cur:
            cur.execute("DELETE FROM memories WHERE conversation_id = %s::uuid", (conversation_id,))
            cur.execute("DELETE FROM messages WHERE conversation_id = %s::uuid", (conversation_id,))
            cur.execute("DELETE FROM conversations WHERE id = %s::uuid", (conversation_id,))
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

In [ ]:
# Example usage (edit COMPANION_ID before running)
if not COMPANION_ID:
    print("Set COMPANION_ID env var or edit the cell to proceed")
else:
    convo = simulate_conversation(
        COMPANION_ID,
        user_profile="Friendly, enjoys hiking and tech talk",
        seed_user_message="Hey! I love hiking on weekends. Any tips?",
        turns=4,
    )
    print("Simulated turns:", len(convo))
    for m in convo:
        print(m["role"].ljust(9), "=>", m["content"][:120])

    #### PERSIST ####
    conversation_id = persist_conversation(COMPANION_ID, EXTERNAL_USER_ID, convo)
    print("Persisted conversation:", conversation_id)

    #### MEMORIES ####
    create_memories_from_messages(COMPANION_ID, conversation_id, convo)
    print("Created memories (best-effort) from messages")

    #### CLEAN (optional) ####
    # delete_conversation_footprint(conversation_id)
    # print('Deleted conversation footprint')